In [68]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.ensemble import GradientBoostingRegressor

In [69]:
df = pd.read_csv(r"C:\Users\kuash\Downloads\archive\Gold Price.csv")

df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

In [70]:
df['Lag1'] = df['Price'].shift(1)
df['Price_Diff'] = df['Price'] - df['Lag1']

df = df.dropna().reset_index(drop=True)

In [71]:
X = df[['Lag1']]
y = df['Price_Diff']

split = int(len(df) * 0.8)

train_df = df.iloc[:split]
test_df = df.iloc[split:]

X_train = train_df[['Lag1']]
y_train = train_df['Price_Diff']

X_test = test_df[['Lag1']]
y_test = test_df['Price_Diff']

actual_price = test_df['Price'].values
lag_test = test_df['Lag1'].values

In [72]:
# Gradient Boosting
gb = GradientBoostingRegressor(random_state=42)
gb.fit(X_train, y_train)
gb_pred = gb.predict(X_test)
gb_final = lag_test + gb_pred

In [73]:
# XGBoost
xgb = XGBRegressor(
    objective='reg:squarederror',
    random_state=42
)
xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_test)
xgb_final = lag_test + xgb_pred

In [74]:
pred_diff = xgb.predict(X_test) 
final_pred = X_test['Lag1'] + pred_diff

In [75]:
rmse_gb = np.sqrt(mean_squared_error(actual_price, gb_final))
mae_gb = mean_absolute_error(actual_price, gb_final)
r2_gb = r2_score(actual_price, gb_final)
mape_gb = np.mean(np.abs((actual_price - gb_final) / actual_price)) * 100

print("\n=== GB MODEL RESULTS ===")
print(f"MAE: {mae_gb:.6f}")
print(f"RMSE: {rmse_gb:.6f}")
print(f"R2: {r2_gb:.6f}")
print(f"MAPE: {mape_gb:.6f}")

rmse_xgb = np.sqrt(mean_squared_error(actual_price, xgb_final))
mae_xgb = mean_absolute_error(actual_price, xgb_final)
r2_xgb = r2_score(actual_price, xgb_final)
mape_xgb = np.mean(np.abs((actual_price - xgb_final) / actual_price)) * 100

print("\n=== XGB MODEL RESULTS ===")
print(f"MAE: {mae_xgb:.6f}")
print(f"RMSE: {rmse_xgb:.6f}")
print(f"R2: {r2_xgb:.6f}")
print(f"MAPE: {mape_xgb:.6f}")


=== GB MODEL RESULTS ===
MAE: 948.164455
RMSE: 1195.599535
R2: 0.996632
MAPE: 1.125514

=== XGB MODEL RESULTS ===
MAE: 597.442307
RMSE: 897.808825
R2: 0.998101
MAPE: 0.687401


In [76]:
import os

os.makedirs("models", exist_ok=True)

In [77]:
import joblib

joblib.dump(gb, "models/gradient_boosting.pkl")
joblib.dump(xgb, "models/xgboost.pkl")

['models/xgboost.pkl']